# 03. Modeling

In [56]:
import json
import sys
from collections import Counter
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import KFold, RepeatedKFold

sys.path.insert(0, str(Path("..").resolve() / "src"))
import evaluate as ev
import models as md

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

PROCESSED_DIR = Path("../data/processed")
FIGURES_DIR = Path("../reports/figures")
RESULTS_DIR = Path("../reports/results")
MODELS_DIR = Path("../models")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

pd.set_option("display.max_columns", 50)

train = pd.read_csv(PROCESSED_DIR / "train_clean.csv")
print(f"train: {train.shape[0]:,} rows x {train.shape[1]} columns")

train: 137 rows x 51 columns


In [57]:
train.head()

,Id,age_days,age_months,open_year,open_month,open_day,open_week,city_freq,city_group_Other,type_IL,type_Other,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,P11,P12,P13,P14,...,P16,P17,P18,P19,P20,P21,P22,P23,P24,P25,P26,P27,P28,P29,P30,P31,P32,P33,P34,P35,P36,P37,all_zero,revenue,log_revenue
0,0,5647,185,1999,7,17,28,0.364964,False,True,False,4,5.0,4.0,4.0,2,2,5,4,5,5,3,5,5.0,1,...,2,2,4,5,4,1,3,3,1,1,1.0,4.0,2.0,3.0,5,3,4,5,5,4,3,4,0,5653753.0,15.547830
1,1,2513,82,2008,2,14,7,0.138686,False,False,False,4,5.0,4.0,4.0,1,2,5,5,5,5,1,5,5.0,0,...,0,0,0,3,2,1,3,2,0,0,0.0,0.0,3.0,3.0,0,0,0,0,0,0,0,0,1,6923131.0,15.750379
2,2,663,21,2013,3,9,10,0.021898,True,True,False,2,4.0,2.0,5.0,2,3,5,5,5,5,2,5,5.0,0,...,0,0,0,1,1,1,1,1,0,0,0.0,0.0,1.0,3.0,0,0,0,0,0,0,0,0,1,2055379.0,14.535971
3,3,1064,34,2012,2,2,5,0.007299,True,True,False,6,4.5,6.0,6.0,4,4,10,8,10,10,8,10,7.5,6,...,9,3,12,20,12,6,1,10,2,2,2.5,2.5,2.5,7.5,25,12,10,6,18,12,12,6,0,2675511.0,14.799651
4,4,2063,67,2009,5,9,19,0.007299,True,True,False,3,4.0,3.0,4.0,2,2,5,5,5,5,2,5,5.0,2,...,2,1,4,2,2,1,2,1,2,3,3.0,5.0,1.0,3.0,5,1,3,2,3,4,3,3,0,4316715.0,15.278005


## 1. Cross-validation strategy

137 rows is too few to trust a single train/validation split — which rows happen to land in
validation can swing RMSE a lot. **Repeated K-fold** (5 folds x 10 repeats, each with a
different random split) reuses every row in many different validation sets, giving a
distribution of RMSE instead of one noisy number. This `outer_cv` is reused for every model in
this notebook (baseline and real models alike) so they're all compared on identical splits.

Real models (§3-§4) also need an `inner_cv` for `GridSearchCV` to pick hyperparameters *inside*
each outer training fold — a plain 5-fold split, reshuffled once (not repeated: it only needs
to be good enough to rank hyperparameter candidates, and it already reruns once per outer fold).

In [58]:
cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f"outer_cv: {cv.get_n_splits(train)} total fold evaluations "
      f"({cv.cvargs['n_splits']} folds x {cv.n_repeats} repeats)")
print(f"inner_cv: {inner_cv.get_n_splits()} folds (for GridSearchCV inside each outer fold)")

outer_cv: 50 total fold evaluations (5 folds x 10 repeats)
inner_cv: 5 folds (for GridSearchCV inside each outer fold)


## 2. Naive baseline: constant prediction, `revenue` vs. `log_revenue` as target

For each fold: take the *training* rows only, compute the mean/median, and score a constant
prediction against the validation rows' actual `revenue` with RMSE. Fitting the constant per
training fold (not once on the full dataset) avoids leaking validation rows into the baseline
itself.

Tried both ways to predict, via the `use_log` flag in `evaluate.cross_val_constant_baseline`:
- **`use_log=True`**: mean/median of `log_revenue`, back-transformed with `expm1` before
  scoring (the project's default so far).
- **`use_log=False`**: mean/median of `revenue` directly — no `log1p`/`expm1` anywhere.

Whichever of the two wins here sets `BASELINE_USE_LOG`, which the rest of the notebook will
reuse so every real model gets tried the same way.

In [59]:
baseline_candidates = []
for use_log in [True, False]:
    for strategy in ["mean", "median"]:
        scores = ev.cross_val_constant_baseline(train, cv, strategy=strategy, use_log=use_log)
        baseline_candidates.append({
            "strategy": strategy,
            "use_log": use_log,
            "target": "log_revenue" if use_log else "revenue",
            "scores": scores,
        })

baseline_scores = {
    f"{c['strategy']} ({c['target']})": c["scores"] for c in baseline_candidates
}

baseline_summary = pd.DataFrame(
    [
        {
            "strategy": c["strategy"],
            "target": c["target"],
            "rmse_mean": c["scores"].mean(),
            "rmse_std": c["scores"].std(),
        }
        for c in baseline_candidates
    ]
).sort_values("rmse_mean").reset_index(drop=True)

baseline_summary

,strategy,target,rmse_mean,rmse_std
0,mean,revenue,2.464601e+06,799564.098326
1,median,revenue,2.483834e+06,863728.067710
2,median,log_revenue,2.483839e+06,863731.686275
3,mean,log_revenue,2.484637e+06,868772.990722


In [60]:
best_baseline = min(baseline_candidates, key=lambda c: c["scores"].mean())
BASELINE_STRATEGY = best_baseline["strategy"]
BASELINE_USE_LOG = best_baseline["use_log"]
BASELINE_RMSE = best_baseline["scores"].mean()
BASELINE_LABEL = f"baseline ({BASELINE_STRATEGY}, {'log' if BASELINE_USE_LOG else 'raw'})"

print(
    f"Reference baseline: predict-the-{BASELINE_STRATEGY} on "
    f"{'log_revenue (back-transformed with expm1)' if BASELINE_USE_LOG else 'revenue directly'} "
    f"-> RMSE = {BASELINE_RMSE:,.0f}"
)

Reference baseline: predict-the-mean on revenue directly -> RMSE = 2,464,601


## 3. Real models: nested CV, one (model, use_log) combination per cell

The grid search below is the slow, crash-prone part — GradientBoosting/RandomForest can take a
while, and there's no reason a bad run of one model should force re-running the others. So
instead of one loop over `models.build_model_specs()`, each combination gets its own cell via
`run_eval.run_combination`, which:

- checkpoints the full nested-CV result (all 50 outer-fold RMSEs, each fold's best
  hyperparameters, and the summary stats) to `reports/results/checkpoints/{model}_{log|raw}.pkl`
  as soon as it finishes;
- **skips straight to loading that file** if it already exists, so re-running a cell — or
  re-running this whole notebook after a restart — never redoes finished work;
- is otherwise independent: run these cells in any order, one at a time, and stop between them
  whenever you like.

`src/run_eval.py` also runs from the command line (`python run_eval.py --model X --use_log
true`, or no args for "everything not yet checkpointed") if you'd rather kick these off outside
the notebook — same checkpoints either way.

In [61]:
import eval_setup as setup
import run_eval as reval

ctx = setup.EvalContext(
    train=train,
    feature_cols=setup.feature_cols(train),
    outer_cv=cv,
    inner_cv=inner_cv,
    baseline={
        "strategy": BASELINE_STRATEGY,
        "use_log": BASELINE_USE_LOG,
        "rmse": BASELINE_RMSE,
        "label": BASELINE_LABEL,
    },
)
print(f"feature_cols ({len(ctx.feature_cols)}): {ctx.feature_cols}")

feature_cols (48): ['age_days', 'age_months', 'open_year', 'open_month', 'open_day', 'open_week', 'city_freq', 'city_group_Other', 'type_IL', 'type_Other', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26', 'P27', 'P28', 'P29', 'P30', 'P31', 'P32', 'P33', 'P34', 'P35', 'P36', 'P37', 'all_zero']


### 3.1 Ridge

In [62]:
ridge_log = reval.run_combination("Ridge", True, ctx)
ridge_log

[skip] Ridge (log): checkpoint already exists at /Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/reports/results/checkpoints/Ridge_log.pkl


{'model': 'Ridge',
 'use_log': True,
 'fold_rmse': array([3462842.67687288, 1835408.5476557 , 1553849.27029006,
        2911666.27823013, 2426548.48570711, 2379618.79426807,
        2736732.28797995, 3264625.44639617, 2183635.47197833,
        1894301.81243349, 1836429.92467103, 1474922.01411147,
        3710972.64337135, 3497262.72232595, 1849025.24870731,
        2979660.4990225 , 1638769.4803837 , 2251855.90186273,
        3432459.72620667, 1989823.76438919, 2102301.41007285,
        1973110.82393416, 1857348.39688381, 4556199.54091117,
        1662078.0286803 , 1633780.62199785, 1688382.54130569,
        3059314.37242282, 3935793.49504672, 3202646.72154136,
        2578724.5638407 , 1580828.01272417, 3055720.50961784,
        1958590.55394449, 3124145.11325106, 3206794.15169803,
        1737484.56276131, 3746409.61157845, 1716643.0192297 ,
        1482264.36034923, 1722186.41653957, 3497208.54910574,
        1394987.93795028, 2920941.33838549, 2641743.09538311,
        1859279.0778

In [63]:
ridge_raw = reval.run_combination("Ridge", False, ctx)
ridge_raw

[skip] Ridge (raw): checkpoint already exists at /Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/reports/results/checkpoints/Ridge_raw.pkl


{'model': 'Ridge',
 'use_log': False,
 'fold_rmse': array([3359391.82730601, 1669199.18530378, 1654206.92942152,
        2814237.32038489, 2429539.3476649 , 2224429.38680202,
        2665375.01991034, 3194405.13273482, 2282566.36939472,
        1870200.04565357, 1651196.54341099, 1600872.98974832,
        3571517.96368242, 3375493.26471733, 1859735.34296608,
        2859764.72217334, 1640067.40974959, 2274570.5096106 ,
        3301389.22981162, 1933517.31460109, 2168820.47244127,
        1810707.57924274, 1694113.62984701, 4372158.75290455,
        1983880.00933245, 1731822.7365464 , 1646463.01238367,
        1623070.49338258, 3775604.6992032 , 3084735.02015603,
        2406755.51521609, 1780308.15313818, 2943431.17131467,
        1907722.0309707 , 3053803.88097948, 2978213.65004657,
        1845139.12364594, 3688012.5102955 , 1740724.36515582,
        1544179.16416484, 1759874.77405376, 3312829.46159269,
        1536938.55997028, 2825832.42365912, 2582692.18781833,
        1760616.766

### 3.2 Lasso

In [64]:
lasso_log = reval.run_combination("Lasso", True, ctx)
lasso_log

[skip] Lasso (log): checkpoint already exists at /Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/reports/results/checkpoints/Lasso_log.pkl


{'model': 'Lasso',
 'use_log': True,
 'fold_rmse': array([3553223.03320838, 1716705.72799311, 1490914.67420495,
        2954053.25925967, 2543636.04369069, 2403205.08004099,
        2823276.83785017, 3209851.25090608, 2097659.57552315,
        1819988.05775702, 1673232.01123827, 1250788.84886027,
        3800255.07569004, 3570326.79891235, 1841479.83055782,
        3099575.5488591 , 1649598.41144339, 2231513.11185437,
        3511812.14934999, 1962547.14109533, 2084014.42578918,
        2064297.06806426, 1769193.4671986 , 4564190.26868576,
        1705860.95415989, 1488560.4195188 , 1683933.69992238,
        1916766.10139916, 3899126.84986464, 3245400.27096037,
        2478624.86696904, 1562297.93368314, 3064100.64112913,
        1887041.11701191, 3171986.51392835, 3356345.26581285,
        1720351.71200533, 4008888.9362724 , 1831972.23889979,
        1576387.84806588, 1760036.6603845 , 3427373.54144775,
        1361756.87587985, 2908385.00883996, 2750912.33452377,
        1868743.3772

In [65]:
lasso_raw = reval.run_combination("Lasso", False, ctx)
lasso_raw

[skip] Lasso (raw): checkpoint already exists at /Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/reports/results/checkpoints/Lasso_raw.pkl


{'model': 'Lasso',
 'use_log': False,
 'fold_rmse': array([ 9267973.51469399,  4458780.61693624,  2724520.80744596,
         3202160.90381799,  4173407.96063664,  2930151.58536986,
         3244306.79091886,  4739740.38177696,  8499634.0132311 ,
         5422159.6475329 ,  3702703.43099683,  6506215.6117022 ,
         4113295.04373389,  4257695.45591304,  3141303.77734283,
         4213085.44439136,  3606096.31753562,  3791622.14753116,
         3492259.38463991, 18579231.58894126,  5499955.49753333,
         4677223.02742352,  3700219.59756918,  4380834.84933065,
         3453185.22324101,  5612329.35031001, 13196534.39312491,
         4359167.69706087,  3948778.56500837,  4490733.26034129,
         3070349.91079553,  3714227.78199116,  3986638.4723226 ,
        13469712.42164727,  3133907.9691441 ,  3902695.81746109,
         9062496.85427334,  4814927.47511437,  2860824.18907297,
         4364004.09995552, 11169564.79812492,  3753548.26603096,
         2665154.3493768 ,  4256392.265

### 3.3 ElasticNet

In [66]:
elasticnet_log = reval.run_combination("ElasticNet", True, ctx)
elasticnet_log

[skip] ElasticNet (log): checkpoint already exists at /Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/reports/results/checkpoints/ElasticNet_log.pkl


{'model': 'ElasticNet',
 'use_log': True,
 'fold_rmse': array([3512953.50971284, 1774378.63384444, 1496743.38500972,
        2963271.97331881, 2405913.92075531, 2396937.69452848,
        2839506.16874348, 3209550.49890751, 2112742.61450273,
        1828565.48805426, 1791200.40892677, 1240896.88547961,
        3800255.07569004, 3570326.79891235, 1829064.29253235,
        3074334.73678915, 1649598.41144339, 2214602.30971541,
        3511812.14934999, 1970737.48596435, 2080340.2657595 ,
        2204293.37262567, 1756678.468603  , 4577612.7018418 ,
        1683462.0204772 , 1497771.91651908, 1655609.36875802,
        2138976.71173772, 3948556.03635721, 3245400.27096037,
        2512477.37194541, 1636803.38394606, 3066743.38422656,
        1914058.41212291, 3099349.6828154 , 3563642.53782776,
        1726107.79035779, 3781385.94928619, 1816453.60739747,
        1576387.84806588, 1753329.10934534, 3441373.6509482 ,
        1285090.90218615, 2913895.21487619, 2750912.33452377,
        1831888

In [67]:
elasticnet_raw = reval.run_combination("ElasticNet", False, ctx)
elasticnet_raw

[skip] ElasticNet (raw): checkpoint already exists at /Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/reports/results/checkpoints/ElasticNet_raw.pkl


{'model': 'ElasticNet',
 'use_log': False,
 'fold_rmse': array([3358469.92978534, 1654103.86001721, 1686186.22561688,
        2817449.34823921, 2446210.42821535, 2195951.16164921,
        2698770.08632615, 3180258.66647007, 2308595.08741094,
        1835380.057208  , 1666118.0866196 , 1633645.05478883,
        3584188.9046701 , 3414795.18318749, 1825949.93513355,
        2852933.12938541, 1645544.35507936, 2287008.94435687,
        3312742.85805644, 1940792.00390768, 2154414.63042281,
        1792681.22622439, 1676027.98231157, 4383238.30161506,
        1935461.37861356, 1763203.16436533, 1645989.08190216,
        1495239.80897446, 3798816.88747932, 3055411.96772403,
        2431842.01338386, 1767079.05712184, 2934315.13596427,
        1922437.43815666, 3062048.96848908, 2981917.00691769,
        1861646.76708505, 3674412.19418477, 1721176.46731696,
        1592290.01442464, 1755813.12149934, 3327593.50642139,
        1563670.73259723, 2807740.25074284, 2690132.23098849,
        177604

### 3.4 RandomForest

In [68]:
randomforest_log = reval.run_combination("RandomForest", True, ctx)
randomforest_log

[skip] RandomForest (log): checkpoint already exists at /Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/reports/results/checkpoints/RandomForest_log.pkl


{'model': 'RandomForest',
 'use_log': True,
 'fold_rmse': array([3370705.30652554, 1721418.86478175, 1604645.970035  ,
        2828149.09957711, 2228067.64473472, 2303732.34349215,
        2806039.40160696, 3020036.61977129, 2120948.15249318,
        1842212.3731729 , 1616623.55650058, 1427709.89820836,
        3491127.94588182, 3267886.82525126, 2082305.95015439,
        2964281.16713894, 1529300.61119514, 2145754.78200761,
        3110189.8697443 , 1977853.68453617, 2060481.3061784 ,
        1631918.11732495, 1771272.58429946, 4304263.90759683,
        1537207.24325416, 1483769.69237254, 1716002.3529157 ,
        1255904.63715108, 3857920.93202718, 3122561.67919207,
        2379784.30018031, 1543275.65442236, 3076632.34993749,
        1912856.91383814, 2904740.86244991, 3044672.97646816,
        1560125.6089828 , 3601648.06341675, 1768778.96079097,
        1376253.36719021, 1603462.05695255, 3298760.96776885,
        1504232.24878792, 2856987.50391572, 2486753.00699346,
        16431

In [69]:
randomforest_raw = reval.run_combination("RandomForest", False, ctx)
randomforest_raw

[skip] RandomForest (raw): checkpoint already exists at /Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/reports/results/checkpoints/RandomForest_raw.pkl


{'model': 'RandomForest',
 'use_log': False,
 'fold_rmse': array([3348308.91742006, 1734573.89399004, 1545097.65404516,
        2823730.31132855, 2443063.65703937, 2256652.9955047 ,
        2780100.43097876, 2983461.5677348 , 2231830.68735859,
        2026887.6694485 , 1795308.46577744, 2182360.55310132,
        3407531.19947385, 3226335.32291685, 1966385.17362645,
        2816424.71056575, 1630494.40982886, 2205226.45853668,
        3067738.69605587, 1943737.55724601, 2134810.21233588,
        1811978.5556613 , 1738283.3245995 , 4112198.13942543,
        1868347.75143643, 1793625.9507811 , 1651174.15865204,
        1546506.43809282, 3675545.19323691, 3111274.85104505,
        2350508.12941719, 1678340.27884076, 2944844.58247736,
        1954522.72823438, 2965096.32565147, 2983015.13579948,
        1750454.0660663 , 3455523.27901035, 1857436.52556401,
        1506351.86722649, 1793256.94035081, 3255945.32508004,
        1676516.48127816, 2885838.11776708, 2356328.91565523,
        1658

### 3.5 GradientBoosting

In [70]:
gradientboosting_log = reval.run_combination("GradientBoosting", True, ctx)
gradientboosting_log

[skip] GradientBoosting (log): checkpoint already exists at /Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/reports/results/checkpoints/GradientBoosting_log.pkl


{'model': 'GradientBoosting',
 'use_log': True,
 'fold_rmse': array([3306378.48800757, 2128193.09324588, 1553767.93073399,
        2989018.83311721, 2356201.50572984, 2368485.80931529,
        2894237.58741972, 3174011.33907155, 2254415.93742195,
        2080202.85769209, 1672217.20059728, 1611467.23781125,
        3601211.17169582, 3348197.58413262, 2101161.74850694,
        3159644.57931089, 1636199.173962  , 2250578.04309771,
        3141572.52090255, 1908528.60601521, 2170271.44604048,
        1683785.59764643, 2039949.49231948, 4311898.5387579 ,
        1568323.04984311, 1565092.79951955, 3175555.73029027,
        1537681.39135696, 3715083.22805421, 3278582.78276627,
        2586168.0646049 , 1909191.80517738, 3004502.4359624 ,
        2029995.22077736, 3172490.44774833, 3182603.76132012,
        1562293.08433947, 3622149.53116187, 1691733.94689296,
        1539213.22541676, 1665998.95478373, 3387682.18187577,
        2237051.08969741, 3047696.7326084 , 2558971.46429233,
        1

In [71]:
gradientboosting_raw = reval.run_combination("GradientBoosting", False, ctx)
gradientboosting_raw

[skip] GradientBoosting (raw): checkpoint already exists at /Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/reports/results/checkpoints/GradientBoosting_raw.pkl


{'model': 'GradientBoosting',
 'use_log': False,
 'fold_rmse': array([3229050.19868667, 1712154.59899025, 1567231.16210983,
        3182838.59947462, 2361475.31296882, 2263976.98126276,
        2903270.72575257, 3119986.60712737, 2211025.33053312,
        1966339.53700782, 1694779.56282563, 1694517.50403745,
        3510474.34487287, 3160456.04770253, 1935812.86783703,
        2968731.74268456, 1694978.74657618, 2269721.10510377,
        3110103.53912026, 1934053.55699631, 2186228.74885928,
        1704502.61797423, 1905103.34954002, 4215053.32768271,
        1637372.43563496, 1628576.14555419, 1506005.86056999,
        1538691.94548491, 3561623.10038578, 3232803.19454467,
        2503756.67391437, 1877587.75062264, 3006331.59147931,
        2271814.40122562, 3081153.67558468, 3124767.80287765,
        1598871.85668773, 3315830.39963576, 1742768.08220009,
        1416296.22824941, 1680377.19663586, 3188880.28795413,
        1845727.70340938, 3067392.85599579, 2375589.0099777 ,
        

### 3.6 GradientBoostingHuber (log only)

Same grid as `GradientBoosting`, but `loss` pinned to `huber` only (see `models.py`) -- a
deliberate alternative to capping the target for the high-revenue outliers, since Huber
down-weights large residuals in the loss without ever discarding the true label the way
capping does. Only `use_log=True` is run here (not a log/raw sweep like the other models) --
this is a single targeted test, not a general model comparison.

In [ ]:
gradientboostinghuber_log = reval.run_combination("GradientBoostingHuber", True, ctx)
gradientboostinghuber_log

### 3.7 RandomForestMAE (log only)

`RandomForestRegressor(criterion="absolute_error")` -- splits on median absolute error
instead of variance, which is inherently less sensitive to a few extreme-revenue rows than
squared error is. A different angle on the same outlier problem as capping (§4) and Huber
(§3.6): here the *whole model* is built around a robust loss, not just the boosting stage.
Grid is trimmed (`max_samples` dropped) since `absolute_error` splits are meaningfully
slower than `squared_error` -- this is a comparison run, not the primary tuning pass.
`use_log=True` only, same as §3.6.

In [ ]:
randomforestmae_log = reval.run_combination("RandomForestMAE", True, ctx)
randomforestmae_log

## 4. Revenue-capped variant (IQR, k=1.5)

Caps `revenue` at `Q3 + 1.5 x IQR` — the same fence `01_eda` used to flag the 8 high-revenue
outliers — **before** computing `log_revenue`, instead of relying on the log transform alone.

Capping happens **inside** the CV loop, per outer fold: `evaluate.nested_cv_grid_search_rmse_capped`
computes Q1/Q3 from each fold's own training rows only (never its validation rows) and caps only
that fold's *training* target before fitting. Validation rows are always scored against the real,
uncapped revenue — scoring against a capped value would just make the metric easier to hit, not
measure real predictive accuracy.

(An earlier pass got this wrong: it capped all 137 rows once with a plain fence and fed the result
straight into CV as a precomputed CSV, which leaked every fold's own validation rows into its
fence and scored against capped values. Those checkpoints have been deleted, and
`build_revenue_variants.py`/the precomputed CSVs are gone — capping is now computed live, inline,
per fold.)

Scoped to `k=1.5` and to `Ridge`/`ElasticNet`/`RandomForest`. Same isolation/checkpoint/resume
mechanics and both `use_log` settings as §3, via `eval_setup.build_context_for_iqr(1.5)` —
checkpoint filenames get an `_iqr1.5` tag so they never collide with §3's uncapped checkpoints.

#### Ridge

In [72]:
ridge_log_iqr1_5 = reval.run_combination("Ridge", True, ctx_iqr1_5)
ridge_log_iqr1_5

[skip] Ridge (log, iqr1.5): checkpoint already exists at /Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/reports/results/checkpoints/Ridge_log_iqr1.5.pkl


{'model': 'Ridge',
 'use_log': True,
 'dataset_tag': 'iqr1.5',
 'iqr_k': 1.5,
 'fold_rmse': array([3488627.31425465, 1774952.92170673, 1551591.22139295,
        2938564.08485154, 2459428.49129549, 2407300.8741871 ,
        2772351.56235842, 3265267.05631659, 2223353.4307759 ,
        1885059.73222977, 1730413.69219502, 1286343.41894764,
        3740058.53104226, 3525440.92875578, 1848493.79504987,
        3019168.78549714, 1658756.59337847, 2267957.2658278 ,
        3458529.60137729, 2027039.4780665 , 2134296.01295004,
        1920545.11755276, 1866513.39004222, 4577811.19177809,
        1603124.31995989, 1453230.32956231, 1713965.79852352,
        1954680.95679201, 3969270.51148758, 3229005.75867327,
        2627695.52225376, 1656646.30157444, 3074368.09426281,
        2069222.28769248, 3142862.90734151, 3223869.00180302,
        1709691.4599468 , 3853809.59195669, 1722652.2286334 ,
        1497936.45121613, 1736373.49728991, 3487176.35159249,
        1303133.09230831, 2928131.1417305

In [73]:
ridge_raw_iqr1_5 = reval.run_combination("Ridge", False, ctx_iqr1_5)
ridge_raw_iqr1_5

[run] Ridge (raw, iqr1.5)
[done] Ridge (raw, iqr1.5): rmse_mean=1,797,147 -> /Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/reports/results/checkpoints/Ridge_raw_iqr1.5.pkl


{'model': 'Ridge',
 'use_log': False,
 'dataset_tag': 'iqr1.5',
 'fold_rmse': array([2019747.22215079, 1624682.44662579, 1620644.03710936,
        1856623.22452724, 1906852.95082117, 2161910.42746812,
        1668397.23559158, 1596251.54418424, 1712113.53819429,
        1687754.4106834 , 1624659.09429602, 1392054.20959605,
        2288832.48127324, 1907246.12507573, 1725827.45802402,
        2021512.45372567, 1542052.7181851 , 1672777.74052591,
        1757999.2952146 , 1870625.36118641, 1606483.80021497,
        1822337.92523352, 1733631.72378565, 2332230.881315  ,
        1628735.1972959 , 1619107.08489185, 1646515.59327393,
        2200021.0871585 , 1974319.7107115 , 2102461.1262803 ,
        1832229.71751311, 1728764.84076078, 2004733.99775674,
        1962175.69074924, 1349378.57864517, 2157675.65923983,
        1778327.49665233, 1972738.30351375, 1586301.43435735,
        1385784.21473764, 1723886.94015936, 1978671.99741539,
        1388794.66758587, 1948240.47199778, 1943962.526

#### ElasticNet

In [74]:
elasticnet_log_iqr1_5 = reval.run_combination("ElasticNet", True, ctx_iqr1_5)
elasticnet_log_iqr1_5

[run] ElasticNet (log, iqr1.5)
[done] ElasticNet (log, iqr1.5): rmse_mean=1,879,572 -> /Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/reports/results/checkpoints/ElasticNet_log_iqr1.5.pkl


{'model': 'ElasticNet',
 'use_log': True,
 'dataset_tag': 'iqr1.5',
 'fold_rmse': array([2441586.48199982, 1720070.65636191, 1520854.37726156,
        1949586.17840636, 1881484.09123786, 2357565.14446667,
        1692880.88836705, 1590898.14845881, 1587974.62506239,
        1783195.8115249 , 1684642.62807368, 1224303.76028232,
        2440093.12900375, 2037387.8937226 , 1740188.003492  ,
        2183655.54084534, 1563969.98632835, 1627294.53895142,
        1928739.68673581, 1891125.99921329, 1509839.5579499 ,
        2067338.91889412, 1774355.33404519, 2575600.85789557,
        1479648.57858021, 1457109.8928347 , 1689110.8247534 ,
        2857646.35829088, 2117508.08152138, 2209256.03499789,
        2001090.57656606, 1886791.77802809, 2101670.15585354,
        1936245.50338708, 1335528.8516969 , 2591181.44548561,
        1715833.63555814, 2190853.86432237, 1696229.62065644,
        1381980.0241312 , 1762388.56239943, 2122718.25082279,
        1292472.21700088, 1847207.13153689, 2035087

In [75]:
elasticnet_raw_iqr1_5 = reval.run_combination("ElasticNet", False, ctx_iqr1_5)
elasticnet_raw_iqr1_5

[run] ElasticNet (raw, iqr1.5)


/Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/tfi-revenue-case/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.311e+13, tolerance: 2.654e+10
  model = cd_fast.enet_coordinate_descent(
/Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/tfi-revenue-case/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.243e+13, tolerance: 2.611e+10
  model = cd_fast.enet_coordinate_descent(
/Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/tfi-revenue-case/

[done] ElasticNet (raw, iqr1.5): rmse_mean=1,791,925 -> /Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/reports/results/checkpoints/ElasticNet_raw_iqr1.5.pkl


/Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/tfi-revenue-case/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.155e+10, tolerance: 2.964e+10
  model = cd_fast.enet_coordinate_descent(


{'model': 'ElasticNet',
 'use_log': False,
 'dataset_tag': 'iqr1.5',
 'fold_rmse': array([1957118.18618779, 1627502.86737212, 1539757.25078051,
        1856111.04317938, 1899416.7294046 , 2169308.06569565,
        1682744.60132585, 1579353.21009541, 1704724.88621402,
        1691372.63386966, 1630107.27407752, 1406196.25369467,
        2297214.4732229 , 1914048.28398795, 1737620.39908836,
        2021168.72847324, 1546448.30376876, 1667944.03827463,
        1766712.44262713, 1871133.74881307, 1606321.01752955,
        1821337.17985019, 1685650.24933459, 2356253.24851669,
        1621838.75493934, 1567934.66838194, 1646135.41946294,
        2128441.86472813, 1992753.90839518, 2105899.17316854,
        1827358.81693333, 1727494.20001864, 1997509.68972413,
        1970533.73396243, 1346140.0834273 , 2118835.63164991,
        1777674.04494884, 1965758.8310135 , 1592414.92472599,
        1418263.93468363, 1733045.27358179, 1978690.24225873,
        1400019.6556584 , 1844434.98957899, 201938

#### RandomForest

In [76]:
randomforest_log_iqr1_5 = reval.run_combination("RandomForest", True, ctx_iqr1_5)
randomforest_log_iqr1_5

[skip] RandomForest (log, iqr1.5): checkpoint already exists at /Users/raqueljolis/Library/Mobile Documents/com~apple~CloudDocs/Projectes/tfi-revenue-case/reports/results/checkpoints/RandomForest_log_iqr1.5.pkl


{'model': 'RandomForest',
 'use_log': True,
 'dataset_tag': 'iqr1.5',
 'fold_rmse': array([1743118.25879325, 1643940.13335452, 1537806.08442559,
        1835811.20464708, 1721328.6507153 , 2186063.59855806,
        1687839.49592269, 1478136.21361573, 1458023.96240313,
        1701864.3845022 , 1553696.74794496, 1252681.34006118,
        2156975.26537881, 1782131.49159257, 2062431.90075419,
        2041755.99201188, 1418685.70623757, 1516298.30832819,
        1775663.00959411, 1736232.69686349, 1487594.74114076,
        1553205.18703409, 1721224.05836051, 2347322.50780939,
        1401677.81630613, 1404281.50316725, 1563198.0319776 ,
        1305967.59437397, 1937931.6614349 , 2214738.69493756,
        1800095.26834453, 1491808.60214636, 1996138.18624231,
        1811393.81693061, 1273874.49137303, 2114858.84011004,
        1513380.00517425, 1840480.98991526, 1561197.32740472,
        1229180.09072889, 1613266.18616123, 1942594.89167204,
        1480916.85543075, 1794904.13964243, 17053

In [ ]:
randomforest_raw_iqr1_5 = reval.run_combination("RandomForest", False, ctx_iqr1_5)
randomforest_raw_iqr1_5

[run] RandomForest (raw, iqr1.5)


## SUMMARY

In [78]:
import aggregate_results as agg

checkpoints = agg.load_checkpoints()
model_summary = agg.build_summary(checkpoints)
model_summary

,model,use_log,dataset_tag,rmse_mean,rmse_std,vs_baseline_pct
0,RandomForest,True,iqr1.5,1.692897e+06,2.642448e+05,-6.831313
1,ElasticNet,False,iqr1.5,1.791925e+06,2.266843e+05,-1.381300
2,Ridge,False,iqr1.5,1.797147e+06,2.292003e+05,-1.093907
3,ElasticNet,True,iqr1.5,1.879572e+06,3.471602e+05,3.442339
4,RandomForest,True,,2.328093e+06,7.741998e+05,-5.538774
5,RandomForest,False,,2.374466e+06,6.754460e+05,-3.657190
6,ElasticNet,False,,2.400130e+06,7.502272e+05,-2.615915
7,GradientBoosting,False,,2.400347e+06,7.293827e+05,-2.607083
8,Ridge,False,,2.400844e+06,7.446242e+05,-2.586927
9,Ridge,True,iqr1.5,2.449877e+06,8.239725e+05,-0.597434


## 6. Per-row predictions: for 04's outlier analysis

`04_results_summary.ipynb` §3 checks whether capping the target at 1.5xIQR causes
systematic underprediction on the true high-revenue outliers, and compares that against
the GradientBoostingHuber alternative from §3.6. That needs every validation row's own
prediction, not just the aggregate RMSE §3/§4 already checkpoint — so these three
cells save that separately, once each, to `reports/results/*.csv`.
`04_results_summary.ipynb` only ever reads these files; it never trains — this is the
only place any of them gets computed. Same cache-check idiom as §3/§4: skip straight to
"already exists" if the CSV is already on disk.

In [79]:
capped_predictions_path = RESULTS_DIR / "randomforest_log_iqr1.5_predictions.csv"

if capped_predictions_path.exists():
    print(f"already exists -> {capped_predictions_path}")
else:
    spec = md.build_model_specs()["RandomForest"]
    capped_predictions = ev.nested_cv_capped_predictions(
        train,
        estimator=spec["estimator"],
        param_grid=spec["param_grid"],
        feature_cols=ctx_iqr1_5.feature_cols,
        outer_cv=ctx_iqr1_5.outer_cv,
        inner_cv=ctx_iqr1_5.inner_cv,
        iqr_k=1.5,
        id_col="Id",
        scale=spec["scale"],
        use_log=True,
    )
    capped_predictions.to_csv(capped_predictions_path, index=False)
    print(f"saved {len(capped_predictions)} (row, fold) predictions -> {capped_predictions_path}")

already exists -> ../reports/results/randomforest_log_iqr1.5_predictions.csv


In [80]:
uncapped_predictions_path = RESULTS_DIR / "randomforest_log_predictions.csv"

if uncapped_predictions_path.exists():
    print(f"already exists -> {uncapped_predictions_path}")
else:
    spec = md.build_model_specs()["RandomForest"]
    uncapped_predictions = ev.nested_cv_predictions(
        train,
        estimator=spec["estimator"],
        param_grid=spec["param_grid"],
        feature_cols=ctx.feature_cols,
        outer_cv=ctx.outer_cv,
        inner_cv=ctx.inner_cv,
        id_col="Id",
        scale=spec["scale"],
        use_log=True,
    )
    uncapped_predictions.to_csv(uncapped_predictions_path, index=False)
    print(f"saved {len(uncapped_predictions)} (row, fold) predictions -> {uncapped_predictions_path}")

AttributeError: module 'evaluate' has no attribute 'nested_cv_predictions'

In [ ]:
gb_huber_predictions_path = RESULTS_DIR / "gradientboostinghuber_log_predictions.csv"

if gb_huber_predictions_path.exists():
    print(f"already exists -> {gb_huber_predictions_path}")
else:
    spec = md.build_model_specs()["GradientBoostingHuber"]
    gb_huber_predictions = ev.nested_cv_predictions(
        train,
        estimator=spec["estimator"],
        param_grid=spec["param_grid"],
        feature_cols=ctx.feature_cols,
        outer_cv=ctx.outer_cv,
        inner_cv=ctx.inner_cv,
        id_col="Id",
        scale=spec["scale"],
        use_log=True,
    )
    gb_huber_predictions.to_csv(gb_huber_predictions_path, index=False)
    print(f"saved {len(gb_huber_predictions)} (row, fold) predictions -> {gb_huber_predictions_path}")